In [9]:
import os
import time
import joblib
import logging
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import MaxAbsScaler
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
from sklearn.cluster import KMeans

logging.basicConfig(level=logging.INFO)


In [10]:
def load_data(input_path):
    logging.info("Loading data...")
    df = pd.read_csv(os.path.join(input_path, 'telephish.csv'),
                     low_memory=False)
    return df


In [11]:
def preprocess_data(df):
    logging.info("Preprocessing data...")

    bool_features = ['is_forwarded', 'is_bot', 'has_media', 'has_text',
                     'is_reply', 'language_match', 'username_matches_language']
    for feature in bool_features:
        df[feature] = df[feature].astype(int)

    # Convert message time to categorical parts of the day
    df['time_of_day'] = df.apply(categorize_time, axis=1)
    df = pd.get_dummies(df, columns=['time_of_day'])

    return df


In [12]:
def categorize_time(row):
    hour = pd.to_datetime(row['time']).hour
    if 5 <= hour < 12:
        return 'Morning'
    elif 12 <= hour < 17:
        return 'Afternoon'
    elif 17 <= hour < 21:
        return 'Night'
    else:
        return 'Midnight'


In [13]:
def prepare_features(df, features):
    logging.info("Preparing features and target variables...")

    initial_row_count = len(df)
    df_dedup = df.drop_duplicates(subset=features)
    dedup_row_count = len(df_dedup)
    rows_dropped = initial_row_count - dedup_row_count
    logging.info(f"Number of duplicate rows dropped: {rows_dropped}")

    X = df_dedup[features]
    y = df_dedup['is_phishing']
    sample_ids = df_dedup["sample_ids"]
    category = df_dedup["category"]

    missing_in_X = X.isnull().sum()
    missing_columns_X = missing_in_X[missing_in_X > 0].index.tolist()
    if missing_columns_X:
        logging.warning(f"Columns with missing values in X: {missing_columns_X}")
        df_clean = df_dedup.dropna(subset=missing_columns_X).reset_index(drop=True)
        clean_row_count = len(df_clean)
        rows_dropped_missing = dedup_row_count - clean_row_count
        logging.info(f"Number of rows dropped due to missing values: {rows_dropped_missing}")
        logging.info(f"Number of rows after dropping missing values: {clean_row_count}")
        X = df_clean[features]
        y = df_clean['is_phishing']
        sample_ids = df_clean["sample_ids"]
        category = df_clean["category"]

    if y.isnull().any():
        missing_rows_y = y[y.isnull()].index.tolist()
        logging.warning(f"Target vector 'y' has missing values in rows: {missing_rows_y}")

    return X, y, sample_ids, category


In [14]:
def build_and_evaluate_model(X, y, sample_ids, category, path_prefix):

    numerical_features = [
        'message_length', 
        'url_count', 
        'total_messages', 
        'unique_users_per_group_message', 
        'messages_repeat_by_user',
        'formatted_text_count'
    ]
    
    param_grid = {
        'n_estimators': [100, 200, 500],
        'max_depth': [None, 5, 10, 20],
        'min_samples_split': [2, 5, 10],
        'class_weight': ['balanced', 'balanced_subsample']
    }
    
    inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
    outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
    
    outer_confusion_matrices = []
    outer_precision_list = []
    outer_recall_list = []
    outer_f1_list = []
    
    outer_sensitivity_list = []
    outer_specificity_list = []
    outer_gmean_list = []
    
    y_true_list = []
    y_pred_list = []
    sample_id_list = []
    fold_data_list = []
    
    # to track best params separately for cluster 0 and cluster 1 across folds
    best_params_cluster0_list = []
    best_params_cluster1_list = []
    best_params_cluster2_list = []
    
    # Collect the max_abs_ from each fold’s scaler (same as original)
    scaler_max_abs_list = []
    
    # -------------------------------------------------------------
    # 1) Outer Cross-Validation
    # -------------------------------------------------------------
    for i, (outer_train_index, outer_test_index) in enumerate(outer_cv.split(X, y)):
        # Split
        X_outer_train = X.iloc[outer_train_index].reset_index(drop=True)
        X_outer_test  = X.iloc[outer_test_index].reset_index(drop=True)
        y_outer_train = y.iloc[outer_train_index].reset_index(drop=True)
        y_outer_test  = y.iloc[outer_test_index].reset_index(drop=True)
        
        sample_ids_outer_test = sample_ids.iloc[outer_test_index].reset_index(drop=True)
        category_outer_test   = category.iloc[outer_test_index].reset_index(drop=True)
    
        # -----------------------------
        # 1.1 Scale the training data
        # -----------------------------
        scaler = MaxAbsScaler()
        scaler.fit(X_outer_train[numerical_features])
        scaler_max_abs_list.append(scaler.max_abs_)

        X_outer_train_scaled = X_outer_train.copy()
        X_outer_test_scaled  = X_outer_test.copy()
        X_outer_train_scaled[numerical_features] = scaler.transform(X_outer_train[numerical_features])
        X_outer_test_scaled[numerical_features]  = scaler.transform(X_outer_test[numerical_features])
    
        # -----------------------------
        # 1.2 K-Means clustering (k=2)
        # -----------------------------
        kmeans = KMeans(n_clusters=3, random_state=0)
        train_clusters = kmeans.fit_predict(X_outer_train_scaled)
        
        # We'll store the trained RF for cluster0 and cluster1
        cluster_models = {0: None, 1: None, 2: None}
        cluster_majority_labels = {}
        
        # -----------------------------
        # 1.3 For each cluster, do inner CV
        # -----------------------------
        for clust in [0, 1, 2]:
            mask_clust = (train_clusters == clust)
            X_clust = X_outer_train_scaled[mask_clust]
            y_clust = y_outer_train[mask_clust]
            
            if len(X_clust) < 10:
                logging.warning(
                        f"Not enough training samples, cluster {clust} in outer fold {i}. "
                        "Skipping this cluster." )
                # If the cluster still has some data, use that cluster's majority label
                if len(X_clust) > 0:
                    cluster_majority_labels[clust] = y_clust.value_counts().idxmax()
                else:
                    # If the cluster is truly empty, fallback to the category majority
                    cluster_majority_labels[clust] = 0
                # We don't train a classifier for this cluster
                continue
            
            # Inner cross-validation grid search
            model_to_tune = RandomForestClassifier()
            rf_gs = GridSearchCV(estimator=model_to_tune, param_grid=param_grid,
                                       cv=inner_cv, n_jobs=-1, scoring='f1_macro')

            rf_gs.fit(X_clust, y_clust)
            best_params = rf_gs.best_params_
            
            # Train final cluster model for this fold
            cluster_rf = RandomForestClassifier(**best_params, random_state=0)
            cluster_rf.fit(X_clust, y_clust)
            cluster_models[clust] = cluster_rf
            
            # Save best params for later aggregation
            if clust == 0:
                best_params_cluster0_list.append(best_params)
            elif clust == 1:
                best_params_cluster1_list.append(best_params)
            else:
                best_params_cluster2_list.append(best_params)
    
        # -----------------------------
        # 1.4 Predict on outer test set
        # -----------------------------
        test_clusters = kmeans.predict(X_outer_test_scaled)
        
        y_pred = []
        for idx, cl in enumerate(test_clusters):
            if cluster_models[cl] is not None:
                instance_pred = cluster_models[cl].predict(X_outer_test_scaled.iloc[[idx]])[0]
            else:
                instance_pred = cluster_majority_labels[cl]
                logging.warning(f"No classifier for cluster {cl} in fold {i}; "f"defaulting to majority label = {instance_pred}.")
            y_pred.append(instance_pred)
    
        # Evaluate confusion matrix, etc.
        cm = confusion_matrix(y_outer_test, y_pred)
        outer_confusion_matrices.append(cm)
    
        TN = cm[0, 0] if cm.shape[0] > 1 else 0
        FP = cm[0, 1] if cm.shape[1] > 1 else 0
        FN = cm[1, 0] if cm.shape[0] > 1 else 0
        TP = cm[1, 1] if cm.shape[1] > 1 else 0
        
        sensitivity = TP / (TP + FN) if (TP + FN) > 0 else 0.0
        specificity = TN / (TN + FP) if (TN + FP) > 0 else 0.0
        gmean = np.sqrt(sensitivity * specificity)
        
        outer_sensitivity_list.append(sensitivity)
        outer_specificity_list.append(specificity)
        outer_gmean_list.append(gmean)

        # Precision, recall, f1 (for each class)
        precision, recall, f1, _ = precision_recall_fscore_support(y_outer_test, y_pred, average=None)
        outer_precision_list.append(precision)
        outer_recall_list.append(recall)
        outer_f1_list.append(f1)
    
        y_true_list.extend(y_outer_test)
        y_pred_list.extend(y_pred)
        sample_id_list.extend(sample_ids_outer_test)
    
        # Print fold confusion matrix
        print(pd.DataFrame(cm,
                           index=['Non-Malicious (Negative)', 'Malicious (Positive)'],
                           columns=['Predicted Non-Malicious', 'Predicted Malicious']))
        # Print classwise precision, recall, F1
        for j, (p, r, f_val) in enumerate(zip(precision, recall, f1)):
            label = 'Malicious (Positive)' if j == 1 else 'Non-Malicious (Negative)'
            print(f"Outer Fold {i+1} Class {label} "
                  f"Precision: {p:.3f}, Recall: {r:.3f}, F1-score: {f_val:.3f}")
        
        # Print sensitivity, specificity, g-mean
        print(f"Sensitivity (TPR): {sensitivity:.3f}")
        print(f"Specificity (TNR): {specificity:.3f}")
        print(f"G-mean: {gmean:.3f}\n")
    
        # Save fold predictions
        fold_data = X_outer_test.copy()
        fold_data['sample_ids'] = sample_ids_outer_test
        fold_data['category'] = category_outer_test
        fold_data['actual'] = y_outer_test
        fold_data['predicted'] = pd.Series(y_pred)
        fold_data.to_csv(f"{path_prefix}random_forest_predictions_fold{i+1}.csv", index=False)
        fold_data_list.append(fold_data)
    
    # ----------------------------------------------------------------
    # 2) After Outer CV: aggregate fold predictions, best params, etc.
    # ----------------------------------------------------------------
    fold_data_all = pd.concat(fold_data_list, axis=0).reset_index(drop=True)
    fold_data_all.to_csv(f"{path_prefix}random_forest_predictions_all.csv", index=False)
    
    # We now have separate best-params for cluster0, cluster1, and cluster2 across folds
    best_params_df_0 = pd.DataFrame(best_params_cluster0_list) if len(best_params_cluster0_list) > 0 else None
    best_params_df_1 = pd.DataFrame(best_params_cluster1_list) if len(best_params_cluster1_list) > 0 else None
    best_params_df_2 = pd.DataFrame(best_params_cluster2_list) if len(best_params_cluster2_list) > 0 else None


    # We’ll pick the mode() of each param as the final hyperparams for each cluster
    def aggregate_params(best_params_df):
        if best_params_df is None or best_params_df.empty:
            return {}
        param_mode = best_params_df.mode().iloc[0].to_dict()
        return param_mode

    best_params_0 = aggregate_params(best_params_df_0)
    best_params_1 = aggregate_params(best_params_df_1)
    best_params_2 = aggregate_params(best_params_df_2)
    
    # Convert possible columns to correct type
    param_types = {
        'n_estimators': int,
        'max_depth': lambda x: int(x) if not pd.isnull(x) else None,
        'min_samples_split': int,
        'criterion': str,
        'max_features': lambda x: x if x is None else str(x),
        'class_weight': lambda x: x if x is None else str(x)
    }
    for param, param_type in param_types.items():
        if param in best_params_0:
            best_params_0[param] = param_type(best_params_0[param])
        if param in best_params_1:
            best_params_1[param] = param_type(best_params_1[param])
        if param in best_params_2:
            best_params_2[param] = param_type(best_params_2[param])
    
    # Aggregate max absolute values from all folds (unchanged)
    scaler_max_abs_array = np.array(scaler_max_abs_list)
    aggregated_max_abs = np.mean(scaler_max_abs_array, axis=0)

    max_abs_df = pd.DataFrame({
        'feature': numerical_features,
        'max_abs': aggregated_max_abs
    })
    max_abs_df.to_csv(f"{path_prefix}scaler_max_abs_values.csv", index=False)

    # ----------------------------------------------------------------
    # 3) Final Scaler + KMeans on entire dataset; Train final cluster models
    # ----------------------------------------------------------------
    final_scaler = MaxAbsScaler()
    final_scaler.fit(X[numerical_features])
    
    joblib.dump(final_scaler, f"{path_prefix}scaler.pkl")
    
    X_scaled = X.copy()
    X_scaled[numerical_features] = final_scaler.transform(X[numerical_features])
    
    # Final KMeans on entire dataset
    final_kmeans = KMeans(n_clusters=3, random_state=0)
    final_train_clusters = final_kmeans.fit_predict(X_scaled)
    
    joblib.dump(final_kmeans, f"{path_prefix}final_kmeans.pkl")
    
    start_time = time.time()
    
    # Train cluster0 final model
    cluster0_mask = (final_train_clusters == 0)
    if cluster0_mask.sum() < 10:
        print("Final: Not enough samples in cluster 0 to train a model.")
    else:
        model_final_0 = RandomForestClassifier(**best_params_0, random_state=0)
        model_final_0.fit(X_scaled[cluster0_mask], y[cluster0_mask])
        joblib.dump(model_final_0, f"{path_prefix}random_forest_model_cluster0.pkl")
    
    # Train cluster1 final model
    cluster1_mask = (final_train_clusters == 1)
    if cluster1_mask.sum() < 10:
        print("Final: Not enough samples in cluster 1 to train a model.")
    else:
        model_final_1 = RandomForestClassifier(**best_params_1, random_state=0)
        model_final_1.fit(X_scaled[cluster1_mask], y[cluster1_mask])
        joblib.dump(model_final_1, f"{path_prefix}random_forest_model_cluster1.pkl")
        
    # Train cluster2 final model
    cluster2_mask = (final_train_clusters == 2)
    if cluster1_mask.sum() < 10:
        print("Final: Not enough samples in cluster 2 to train a model.")
    else:
        model_final_2 = RandomForestClassifier(**best_params_2, random_state=0)
        model_final_2.fit(X_scaled[cluster2_mask], y[cluster2_mask])
        joblib.dump(model_final_2, f"{path_prefix}random_forest_model_cluster2.pkl")
    
    end_time = time.time()
    duration = end_time - start_time
    hours, rem = divmod(duration, 3600)
    minutes, seconds = divmod(rem, 60)
    print(f"\nTotal Execution Time: {int(hours)}h {int(minutes)}m {seconds:.2f}s")
    
    # Save final cluster hyperparams
    with open(f"{path_prefix}random_forest_model_params.txt", 'w') as f:
        f.write("Cluster 0 final params:\n")
        f.write(str(best_params_0) + "\n\n")
        f.write("Cluster 1 final params:\n")
        f.write(str(best_params_1) + "\n")
        f.write("Cluster 2 final params:\n")
        f.write(str(best_params_2) + "\n")

    # ----------------------------------------------------------------
    # 4) Report the average metrics across folds (unchanged from original)
    # ----------------------------------------------------------------
    average_precision = np.mean(outer_precision_list, axis=0)
    average_recall    = np.mean(outer_recall_list, axis=0)
    average_f1        = np.mean(outer_f1_list, axis=0)
    
    std_precision = np.std(outer_precision_list, axis=0)
    std_recall    = np.std(outer_recall_list, axis=0)
    std_f1        = np.std(outer_f1_list, axis=0)

    print("\nAverage and Standard Deviation of Precision, Recall, and F1-score Across All Folds:")
    for j in range(len(average_precision)):
        label = 'Malicious (Positive)' if j == 1 else 'Non-Malicious (Negative)'
        print(
            f"Class {label} - Precision: {average_precision[j]:.3f} ± {std_precision[j]:.3f}, "
            f"Recall: {average_recall[j]:.3f} ± {std_recall[j]:.3f}, "
            f"F1-score: {average_f1[j]:.3f} ± {std_f1[j]:.3f}"
        )
        
    avg_sensitivity = np.mean(outer_sensitivity_list)
    std_sensitivity = np.std(outer_sensitivity_list)
    avg_specificity = np.mean(outer_specificity_list)
    std_specificity = np.std(outer_specificity_list)
    avg_gmean       = np.mean(outer_gmean_list)
    std_gmean       = np.std(outer_gmean_list)
    
    print("\nAverage and Standard Deviation of Sensitivity, Specificity, and G-mean Across All Folds:")
    print(f"Sensitivity: {avg_sensitivity:.3f} ± {std_sensitivity:.3f}")
    print(f"Specificity: {avg_specificity:.3f} ± {std_specificity:.3f}")
    print(f"G-mean: {avg_gmean:.3f} ± {std_gmean:.3f}")


In [15]:
def main():
    save_path = '3cluster-results/'
    input_path = '../../../data/'

    os.makedirs(save_path, exist_ok=True)
    df = load_data(input_path)
    df = preprocess_data(df)

    # Example feature set 
    selected_features = [
        'is_bot', 'message_length', 'has_media', 'unique_users_per_group_message',
        'is_reply', 'url_count', 'total_messages', 'normalized_days_until_first_post',
        'language_match', 'messages_repeat_by_user', 'username_matches_language', 
        'formatted_text_count'
    ]

    X_selected, y_selected, sample_ids_selected, category_selected = prepare_features(df, selected_features)

    print(np.unique(y_selected, return_counts=True))
    build_and_evaluate_model(X_selected, y_selected, sample_ids_selected, category_selected, save_path)


In [16]:
if __name__ == '__main__':
    main()

INFO:root:Loading data...
INFO:root:Preprocessing data...
INFO:root:Preparing features and target variables...
INFO:root:Number of duplicate rows dropped: 7315
INFO:root:Number of rows dropped due to missing values: 1017
INFO:root:Number of rows after dropping missing values: 49877


(array([False,  True]), array([48880,   997]))
                          Predicted Non-Malicious  Predicted Malicious
Non-Malicious (Negative)                     9739                   37
Malicious (Positive)                           64                  136
Outer Fold 1 Class Non-Malicious (Negative) Precision: 0.993, Recall: 0.996, F1-score: 0.995
Outer Fold 1 Class Malicious (Positive) Precision: 0.786, Recall: 0.680, F1-score: 0.729
Sensitivity (TPR): 0.680
Specificity (TNR): 0.996
G-mean: 0.823
                          Predicted Non-Malicious  Predicted Malicious
Non-Malicious (Negative)                     9748                   28
Malicious (Positive)                           56                  144
Outer Fold 2 Class Non-Malicious (Negative) Precision: 0.994, Recall: 0.997, F1-score: 0.996
Outer Fold 2 Class Malicious (Positive) Precision: 0.837, Recall: 0.720, F1-score: 0.774
Sensitivity (TPR): 0.720
Specificity (TNR): 0.997
G-mean: 0.847
                          Predicted